# Mapa de visitas Â· Ciudad ParaÃ­so (CDO Â· AlcaldÃ­a de Santiago de Cali)
Notebook autocontenido: toma los 5 Excel del corte + los polÃ­gonos de manzanas, calcula los cruces, la clasificaciÃ³n en 4 grupos y el semÃ¡foro por manzana, y **dibuja el mapa aquÃ­ mismo**. TambiÃ©n genera los Excel y un HTML del mapa para descargar.

**CÃ³mo usar:** ejecuta las celdas en orden (Entorno â†’ Ejecutar todo). En la celda 3 sube los 5 `.xlsx` del corte y `MANZANAS_U_2024.geojson`.
âš ï¸ Contiene datos personales: Ãºsalo solo en una cuenta/entorno autorizado y no compartas el notebook con datos cargados.

### 1. Entorno y cÃ³digo del ETL

In [ ]:
import os, json
os.makedirs('config', exist_ok=True)
os.makedirs('datos/raw/poligonos_ciudad_paraiso', exist_ok=True)
os.environ['LOCAL_DIR'] = os.path.abspath('datos')   # carpeta que imita el bucket (raw/ y curated/)
print('Listo')

In [ ]:
# Equivalencia cÃ³digo de manzana <-> id catastral (editable) y polÃ­gono de la zona
open('config/manzanas.json','w',encoding='utf-8').write(r'''{
  "_nota": "Equivalencia entre el cÃ³digo interno de manzana (tabla de responsables) y el IDMANZANA catastral de MANZANAS_U_2024.geojson. Cruzada por posiciÃ³n/forma contra el plano de manzanas A0157-A0197. 202 y 203 quedan SIN id catastral: confirmar (hipÃ³tesis: 03100011 y 03100018). Editable sin redesplegar: subir una copia a config/manzanas.json del bucket.",
  "manzanas": [
    {"n": 1,  "codigo": 157, "id": "03060006", "responsable": "Bienestar Social - AndrÃ©s Felipe NiÃ±o"},
    {"n": 2,  "codigo": 158, "id": "03060009", "responsable": "UAEPA-DADII"},
    {"n": 3,  "codigo": 159, "id": "03060021", "responsable": "DATIC"},
    {"n": 4,  "codigo": 160, "id": "03060022", "responsable": "Deporte y la RecreaciÃ³n"},
    {"n": 5,  "codigo": 161, "id": "03060008", "responsable": "UAESP"},
    {"n": 6,  "codigo": 162, "id": "03060007", "responsable": "UAESP"},
    {"n": 7,  "codigo": 163, "id": "03090001", "responsable": "Predio del PMU"},
    {"n": 8,  "codigo": 164, "id": "03090007", "responsable": "Bienestar Social - EducaciÃ³n"},
    {"n": 9,  "codigo": 165, "id": "03090015", "responsable": "Seguridad y Justicia - Riesgos"},
    {"n": 10, "codigo": 166, "id": "03090014", "responsable": "Gobierno"},
    {"n": 11, "codigo": 167, "id": "03090008", "responsable": "EDRU Â· adquisiciÃ³n predial"},
    {"n": 12, "codigo": 168, "id": "03090002", "responsable": "Conjunto residencial Ãgora"},
    {"n": 13, "codigo": 169, "id": "03090003", "responsable": "EDRU Â· adquisiciÃ³n predial"},
    {"n": 14, "codigo": 170, "id": "03090006", "responsable": "Etnias"},
    {"n": 15, "codigo": 171, "id": "03090009", "responsable": "EstaciÃ³n de PolicÃ­a Fray DamiÃ¡n"},
    {"n": 16, "codigo": 172, "id": "03090012", "responsable": "SecretarÃ­a de Vivienda"},
    {"n": 17, "codigo": 173, "id": "03090013", "responsable": "Turismo"},
    {"n": 18, "codigo": 174, "id": "03090011", "responsable": "Paz y Cultura Ciudadana"},
    {"n": 19, "codigo": 175, "id": "03090010", "responsable": "ParticipaciÃ³n Ciudadana (presidente JAC)"},
    {"n": 20, "codigo": 176, "id": "03090005", "responsable": "Paz y Cultura Ciudadana"},
    {"n": 21, "codigo": 177, "id": "03090004", "responsable": "Seguridad y Justicia"},
    {"n": 22, "codigo": 194, "id": "03070017", "responsable": "Desarrollo EconÃ³mico - Movilidad"},
    {"n": 23, "codigo": 195, "id": "03070018", "responsable": "Bienes - ParticipaciÃ³n Ciudadana (TIO)"},
    {"n": 24, "codigo": 196, "id": "03100019", "responsable": "Cultura"},
    {"n": 25, "codigo": 197, "id": "03100010", "responsable": "Desarrollo EconÃ³mico"},
    {"n": 26, "codigo": 202, "id": null,       "responsable": "BÃºnker de la FiscalÃ­a"},
    {"n": 27, "codigo": 203, "id": null,       "responsable": "SecretarÃ­a de Vivienda"}
  ]
}
''')
open('config/poligono_ciudad_paraiso_wgs84.geojson','w',encoding='utf-8').write(r'''{"type": "Feature", "properties": {"name": "Ciudad Para\u00edso"}, "geometry": {"type": "Polygon", "coordinates": [[[-76.52660898738255, 3.4446436150156408], [-76.52610728175726, 3.443202849991695], [-76.52600377739127, 3.4425748724649075], [-76.52865612339545, 3.4427612459305], [-76.53083097914715, 3.4426891534078172], [-76.53197010710772, 3.4424450550378842], [-76.53202405502302, 3.442543331237526], [-76.53211556179104, 3.4435307353276086], [-76.53221515562474, 3.44440870018517], [-76.53258960645213, 3.4469191016319334], [-76.52986829384582, 3.447686719687869], [-76.52744683252324, 3.447910194702635], [-76.52660898738255, 3.4446436150156408]]]}}''')
print('config ok')

In [ ]:
# Equivalente a la magic '%%writefile etl_handler.py' (compatible con
# IPython, Jupyter, Colab y Python puro). El cuerpo del modulo va en base64
# para preservarlo intacto; se decodifica y se escribe a etl_handler.py.
import base64 as _b64
_ETL_B64 = (
    "..."
)
_ETL_SRC = _b64.b64decode(_ETL_B64).decode('utf-8')
with open('etl_handler.py', 'w', encoding='utf-8') as _f:
    _f.write(_ETL_SRC)
print('etl_handler.py escrito (%d bytes)' % len(_ETL_SRC))


### 2. Cargar los datos del corte

In [ ]:
import shutil, glob

CORTE = 'data_manual_corte_14_oct_2026'   # cambia la fecha segun el corte (formato data_manual_corte_DD_mes_AAAA)
RAW = f'datos/raw/{CORTE}'
POLY = 'datos/raw/poligonos_ciudad_paraiso'
os.makedirs(RAW, exist_ok=True)
os.makedirs(POLY, exist_ok=True)

# Carpeta local del proyecto con los datos de origen.
# El notebook esta en notebooks/, por lo que la carpeta data/ esta un nivel arriba.
# Se puede sobreescribir con la variable de entorno DATA_DIR.
DATA_DIR = os.environ.get('DATA_DIR')
if not DATA_DIR:
    candidatos = [os.path.join('..', 'data'), 'data',
                  r'C:\Users\Jorge\Desktop\mapa_Analitico_ciudad_Paraiso\data']
    DATA_DIR = next((p for p in candidatos if os.path.isdir(p)), candidatos[0])
DATA_DIR = os.path.abspath(DATA_DIR)
GEOJSON_DIR = os.path.join(DATA_DIR, 'geojson')

if os.path.isdir(DATA_DIR):
    # Entorno local: copiar los .xlsx y el .geojson desde data/ a la estructura raw/.
    xlsx = glob.glob(os.path.join(DATA_DIR, '*.xlsx'))
    for src in xlsx:
        shutil.copy2(src, os.path.join(RAW, os.path.basename(src)))
    geo = glob.glob(os.path.join(GEOJSON_DIR, 'MANZANAS_U_2024.geojson')) \
          or glob.glob(os.path.join(DATA_DIR, '**', 'MANZANAS_U_2024.geojson'), recursive=True)
    for src in geo:
        shutil.copy2(src, os.path.join(POLY, os.path.basename(src)))
    print(f'Origen de datos: {DATA_DIR}')
    print(f'Copiados {len(xlsx)} .xlsx a {RAW}')
    print(f'Copiados {len(geo)} .geojson a {POLY}')
else:
    # Colab u otro entorno sin la carpeta data/: subir los archivos manualmente.
    try:
        from google.colab import files
        print('Sube los 5 .xlsx del corte y MANZANAS_U_2024.geojson')
        up = files.upload()
        for name, data in up.items():
            dest = POLY if name.lower().endswith('.geojson') else RAW
            open(os.path.join(dest, name), 'wb').write(data)
    except ImportError:
        print('No se encontro la carpeta data/. Define DATA_DIR o copia los archivos a', RAW, 'y a', POLY)

print('raw:', sorted(os.listdir(RAW)))
print('poligonos:', sorted(os.listdir(POLY)))

### 3. Ejecutar cruces y clasificaciÃ³n

In [ ]:
import importlib, etl_handler
importlib.reload(etl_handler)
store = etl_handler.Store()
res = etl_handler.run(store, CORTE)
fecha = res['fecha']
print('ClasificaciÃ³n:', json.dumps(res['clasificacion'], ensure_ascii=False, indent=1))
print('UbicaciÃ³n:', res['ubicacion'], '| con manzana:', res['con_manzana'], '| grilla:', res['grilla'])

### 4. Mapa interactivo

In [ ]:
HTML_TPL = r'''<!DOCTYPE html>
<html lang="es"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Mapa de visitas Â· Ciudad ParaÃ­so</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<link rel="stylesheet" href="https://unpkg.com/leaflet.markercluster@1.5.3/dist/MarkerCluster.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet.markercluster@1.5.3/dist/leaflet.markercluster.js"></script>
<style>
:root{--bg:#0f1e3a;--fg:#fff;--mut:#9fb0cf;--acc:#c9992d;--r:#e34948;--a:#eda100;--v:#1baf7a;--line:rgba(255,255,255,.14)}
*{box-sizing:border-box;margin:0;padding:0}
html,body{height:100%;font-family:system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;font-size:14px}
#map{position:absolute;inset:0}
#top{position:absolute;z-index:1000;top:0;left:0;right:0;display:flex;gap:10px;align-items:center;padding:8px 12px;background:rgba(15,30,58,.94);color:var(--fg);padding-left:56px}
#top h1{font-size:15px;font-weight:650;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
#top .sub{font-size:11px;color:var(--acc);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
#top .sp{flex:1}
#top button{background:transparent;color:var(--fg);border:1px solid var(--line);border-radius:8px;padding:6px 10px;font-size:12px;cursor:pointer}
#panel{position:absolute;z-index:1000;top:52px;right:10px;width:320px;max-height:calc(100% - 64px);overflow:auto;background:rgba(15,30,58,.95);color:var(--fg);border-radius:12px;padding:12px 14px;box-shadow:0 6px 24px rgba(0,0,0,.35)}
#panel h3{font-size:11px;letter-spacing:.06em;text-transform:uppercase;color:var(--acc);margin:12px 0 6px}
#panel h3:first-child{margin-top:0}
.chips{display:flex;flex-wrap:wrap;gap:6px}
.chip{display:inline-flex;align-items:center;gap:6px;padding:5px 9px;border:1px solid var(--line);border-radius:999px;cursor:pointer;user-select:none;font-size:12px}
.chip input{display:none}
.chip i{width:11px;height:11px;border-radius:50%;display:inline-block;border:1px solid rgba(255,255,255,.5)}
.chip.off{opacity:.38}
select,input[type=search]{width:100%;padding:7px 9px;border-radius:8px;border:1px solid var(--line);background:rgba(255,255,255,.08);color:var(--fg);font-size:13px}
select option{color:#000}
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:6px;text-align:center}
.kpi{border-radius:8px;padding:6px 2px;background:rgba(255,255,255,.07)}
.kpi b{display:block;font-size:17px}.kpi span{font-size:10px;color:var(--mut)}
.row{display:flex;gap:8px;align-items:center;font-size:12px;color:var(--mut);margin-top:6px}
.small{font-size:11px;color:var(--mut);line-height:1.45}
.lab{background:rgba(15,30,58,.85)!important;color:#fff!important;border:0!important;box-shadow:none!important;font-weight:700;font-size:10px;padding:1px 4px!important}
.lab:before{display:none!important}
.leaflet-popup-content{font-size:12.5px;line-height:1.5;margin:10px 12px;min-width:210px}
.pp b.t{font-size:13.5px}
.cl{width:34px;height:34px;border-radius:50%;background:#2a78d6;color:#fff;border:3px solid rgba(255,255,255,.85);display:flex;align-items:center;justify-content:center;font-weight:700;font-size:12px;box-shadow:0 1px 6px rgba(0,0,0,.4)}
.tag{display:inline-block;padding:1px 8px;border-radius:999px;color:#fff;font-size:11px;font-weight:700}
#tog{display:none}
@media (max-width:760px){
 #panel{top:auto;bottom:0;left:0;right:0;width:auto;max-height:58%;border-radius:14px 14px 0 0;transform:translateY(calc(100% - 0px));transition:transform .25s}
 #panel.open{transform:none}
 #tog{display:inline-block}
 #top h1{font-size:13px}
 .kpis{grid-template-columns:repeat(4,1fr)}
}
</style></head>
<body>
<div id="map"></div>
<div id="top">
 <div style="min-width:0"><h1>Visitas Â· Ciudad ParaÃ­so</h1><div class="sub" id="sub">Cargandoâ€¦</div></div>
 <div class="sp"></div>
 <button id="tog">Filtros</button>
 <button id="fit">Ver zona</button>
</div>
<div id="panel">
 <h3>Resumen (con filtros)</h3>
 <div class="kpis"><div class="kpi"><b id="k-t">0</b><span>en mapa</span></div><div class="kpi"><b id="k-r" style="color:#ff8a89">0</b><span>rojo</span></div><div class="kpi"><b id="k-a" style="color:#ffd05c">0</b><span>amarillo</span></div><div class="kpi"><b id="k-v" style="color:#5fe0b0">0</b><span>verde</span></div></div>
 <div class="small" id="k-x" style="margin-top:6px"></div>
 <h3>SemÃ¡foro</h3>
 <div class="chips" id="c-sem"></div>
 <h3>UbicaciÃ³n</h3>
 <div class="chips" id="c-ub"></div>
 <h3>Manzana</h3>
 <select id="f-mz"></select>
 <h3>Buscar</h3>
 <input id="f-q" type="search" placeholder="Nombre, documento o direcciÃ³n">
 <h3>Color de la manzana</h3>
 <select id="f-mode"><option value="peor">Peor caso (rojo si hay alguien sin visita)</option><option value="pred">Predominante (color con mÃ¡s personas)</option></select>
 <h3>Leyenda</h3>
 <div class="small"><b style="color:#ff8a89">Rojo</b>: falta visitar totalmente Â· <b style="color:#ffd05c">Amarillo</b>: parcial / no completo Â· <b style="color:#5fe0b0">Verde</b>: completo.<br>Punto lleno = GPS (donde se llenÃ³ el formulario). Punto con borde punteado = ubicaciÃ³n aproximada por direcciÃ³n.<br>Las personas sin ubicaciÃ³n no aparecen en el mapa.</div>
</div>
<script>
const COL={ROJO:'#e34948',AMARILLO:'#eda100',VERDE:'#1baf7a','SIN DATOS':'#8893a8'};
const esc=s=>String(s??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const map=L.map('map',{zoomControl:true}).setView([3.4447,-76.529],17);
const esri=(l,n)=>L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/'+l+'/MapServer/tile/{z}/{y}/{x}',{maxZoom:19,attribution:'Tiles &copy; Esri | CDO AlcaldÃ­a de Cali'});
const base={
 'Calles (Esri)':esri('World_Street_Map'),
 'SatÃ©lite (Esri)':esri('World_Imagery'),
 'TopogrÃ¡fico (Esri)':esri('World_Topo_Map'),
 'OpenStreetMap (solo vÃ­a URL https)':L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png',{maxZoom:19,attribution:'&copy; OpenStreetMap | CDO AlcaldÃ­a de Cali'})
};
base['Calles (Esri)'].addTo(map);
L.control.layers(base,null,{position:'bottomleft',collapsed:true}).addTo(map);
map.zoomControl.setPosition('bottomleft');
const clus=L.markerClusterGroup({maxClusterRadius:28,spiderfyOnMaxZoom:true,showCoverageOnHover:false,disableClusteringAtZoom:19,iconCreateFunction:c=>L.divIcon({html:`<div class="cl">${c.getChildCount()}</div>`,className:'',iconSize:[34,34]})});
const mzLayer=L.layerGroup().addTo(map), zona=L.layerGroup().addTo(map);
map.addLayer(clus);
const st={sem:new Set(['ROJO','AMARILLO','VERDE']),ub:new Set(['GPS','APROX']),mz:'',q:'',mode:'peor'};
let DATA=[],MZ=null,BOUNDS=null;
function chips(el,items,set,onchg){el.innerHTML='';items.forEach(([k,lab,c])=>{const l=document.createElement('label');l.className='chip';l.innerHTML=`<input type="checkbox" checked>${c?`<i style="background:${c}"></i>`:''}${lab}`;l.querySelector('input').onchange=e=>{e.target.checked?set.add(k):set.delete(k);l.classList.toggle('off',!e.target.checked);onchg()};el.appendChild(l)})}
function mzColor(p){if(!p.total)return COL['SIN DATOS'];return COL[st.mode==='peor'?p.peor:p.predominante]}
function drawMz(){mzLayer.clearLayers();MZ.features.forEach(f=>{const p=f.properties,c=mzColor(p);
 const l=L.geoJSON(f,{style:{color:c,weight:2,fillColor:c,fillOpacity:.22}}).addTo(mzLayer);
 l.bindTooltip(p.codigo,{permanent:true,direction:'center',className:'lab'});
 l.bindPopup(`<div class="pp"><b class="t">Manzana ${esc(p.codigo)}</b><br><small>Cat. ${esc(p.id)}</small><br>Responsable: <b>${esc(p.responsable)}</b><br>Personas ubicadas: <b>${p.total}</b><br><span class="tag" style="background:${COL.ROJO}">Rojo ${p.rojo}</span> <span class="tag" style="background:${COL.AMARILLO}">Amarillo ${p.amarillo}</span> <span class="tag" style="background:${COL.VERDE}">Verde ${p.verde}</span></div>`);
 l.on('click',()=>{});})}
function popup(d){const c=COL[d.sem];return `<div class="pp"><b class="t">${esc(d.n)}</b><br><span class="tag" style="background:${c}">${esc(d.estado)}</span><br>
 Doc: <b>${esc(d.doc)}</b>${d.tel?` Â· Tel: <b>${esc(d.tel)}</b>`:''}<br>Barrio: ${esc(d.b||'â€”')}<br>DirecciÃ³n: ${esc(d.dir||'â€”')}<br>
 Manzana: <b>${esc(d.mz||'â€”')}</b>${d.mzr?` Â· ${esc(d.mzr)}`:''}<br>
 Ficha: ${esc(d.f)} Â· Hogares: ${esc(d.h)} Â· Salud: ${esc(d.sa)}<br>${d.v>0?`Visitas documentadas: ${d.v}<br>`:''}
 <b>QuÃ© hacer:</b> ${esc(d.ac)}<br>${d.fa?`Falta: ${esc(d.fa)}<br>`:''}
 <small>${d.u==='GPS'?'UbicaciÃ³n GPS (referencial)':'UbicaciÃ³n aproximada por direcciÃ³n'}${d.al?' Â· âš  '+esc(d.al):''}</small><br>
 <a target="_blank" rel="noopener" href="https://www.google.com/maps?q=${d.la},${d.lo}">Abrir en Google Maps</a></div>`}
function render(){clus.clearLayers();const q=st.q.trim().toLowerCase();let n=0,r=0,a=0,v=0;
 DATA.forEach(d=>{const ok=st.sem.has(d.sem)&&st.ub.has(d.u)&&(!st.mz||d.mz===st.mz)&&(!q||(d.n+' '+d.doc+' '+(d.dir||'')).toLowerCase().includes(q));
  if(!ok)return;n++;d.sem==='ROJO'?r++:d.sem==='AMARILLO'?a++:v++;
  if(!d._m){d._m=L.circleMarker([d.la,d.lo],{radius:7,weight:2,color:'#fff',fillColor:COL[d.sem],fillOpacity:.95,dashArray:d.u==='GPS'?null:'3 2'});d._m.bindPopup(()=>popup(d),{maxWidth:300})}
  clus.addLayer(d._m)});
 k_t.textContent=n;k_r.textContent=r;k_a.textContent=a;k_v.textContent=v}
const $=id=>document.getElementById(id);const k_t=$('k-t'),k_r=$('k-r'),k_a=$('k-a'),k_v=$('k-v');
function fitZone(){if(BOUNDS)map.fitBounds(BOUNDS,{padding:[30,30]})}
Promise.resolve(__DATA__).then(j=>{
 DATA=j.puntos;MZ=j.manzanas;
 $('sub').textContent=`Corte ${j.corte} Â· ${j.resumen.total} personas Â· ${j.puntos.length} ubicadas Â· ${j.resumen.sin_ubicacion} sin ubicaciÃ³n`;
 $('k-x').textContent=`Sin ubicaciÃ³n (no se grafican): ${j.resumen.sin_ubicacion}. Con manzana asignada: ${j.resumen.con_manzana}.`;
 chips($('c-sem'),[['ROJO','Rojo',COL.ROJO],['AMARILLO','Amarillo',COL.AMARILLO],['VERDE','Verde',COL.VERDE]],st.sem,render);
 chips($('c-ub'),[['GPS','GPS',null],['APROX','Aproximada',null]],st.ub,render);
 const sel=$('f-mz');sel.innerHTML='<option value="">Todas las manzanas</option>';
 MZ.features.map(f=>f.properties).sort((a,b)=>a.codigo.localeCompare(b.codigo)).forEach(p=>{const o=document.createElement('option');o.value=p.codigo;o.textContent=`${p.codigo} Â· ${p.responsable} (${p.total})`;sel.appendChild(o)});
 sel.onchange=()=>{st.mz=sel.value;render();if(st.mz){const f=MZ.features.find(f=>f.properties.codigo===st.mz);if(f)map.fitBounds(L.geoJSON(f).getBounds(),{padding:[40,40],maxZoom:19})}};
 $('f-q').oninput=e=>{st.q=e.target.value;render()};
 $('f-mode').onchange=e=>{st.mode=e.target.value;drawMz()};
 if(j.zona)L.geoJSON(j.zona,{style:{color:'#c9992d',weight:2,dashArray:'6 6',fill:false}}).addTo(zona);
 drawMz();render();BOUNDS=L.geoJSON(MZ).getBounds();fitZone();setTimeout(()=>{map.invalidateSize();fitZone()},400);setTimeout(()=>{map.invalidateSize();fitZone()},1500);
}).catch(e=>{$('sub').textContent='No se pudo cargar la data ('+e.message+')'});
if(window.ResizeObserver)new ResizeObserver(()=>map.invalidateSize()).observe($('map'));window.addEventListener('resize',()=>map.invalidateSize());
$('fit').onclick=fitZone;$('tog').onclick=()=>$('panel').classList.toggle('open');
</script></body></html>
'''

In [ ]:
import html as _h, warnings; warnings.filterwarnings("ignore")
from IPython.display import HTML, display
data = open(f'datos/curated/visitas/{fecha}/mapa_data.json', encoding='utf-8').read()
page = HTML_TPL.replace('__DATA__', data.replace('</', '<\\/'))
open('Mapa_Visitas_Ciudad_Paraiso.html', 'w', encoding='utf-8').write(page)
display(HTML(f'<iframe srcdoc="{_h.escape(page, quote=True)}" style="width:100%;height:780px;border:0;display:block;border-radius:8px"></iframe>'))

### 5. Descargar resultados

In [ ]:
import shutil
out = f'datos/curated'
for f in [f'{out}/visitas/{fecha}/Semaforo_Manzanas.xlsx', f'{out}/visitas/{fecha}/Hoja_de_Ruta_Visitas.xlsx', f'{out}/cruces/{fecha}/Consolidado_Operativo.xlsx', 'Mapa_Visitas_Ciudad_Paraiso.html']:
    print(f)
try:
    from google.colab import files
    for f in [f'{out}/visitas/{fecha}/Semaforo_Manzanas.xlsx', f'{out}/visitas/{fecha}/Hoja_de_Ruta_Visitas.xlsx', f'{out}/cruces/{fecha}/Consolidado_Operativo.xlsx', 'Mapa_Visitas_Ciudad_Paraiso.html']:
        files.download(f)
except ImportError:
    pass